# 🔎 OPERAÇÃO CHECAGEM
### Simulado de NLP 2 · Trilha de Linguagem Natural · preparação para a OIAA 2026

---

> **"Uma mentira atravessa o país enquanto a gente ainda está ligando para a fonte. Não
> quero uma máquina que decida o que é verdade. Quero uma que me diga o que ler primeiro."**
>
> — Lúcia Fernandes, editora-chefe de uma agência de checagem

## 📨 Pauta urgente

**De:** Lúcia Fernandes · Agência de checagem de fatos
**Para:** Equipe de Linguagem Natural

> Recebemos por dia milhares de links, prints e textos encaminhados pelos leitores. A
> equipe checa, bem checado, umas vinte histórias. O resto fica na fila.
>
> Existe um acervo **real e público**, o **Fake.br**: notícias falsas que circularam no
> Brasil entre 2016 e 2018, cada uma **pareada com uma notícia verdadeira do mesmo
> assunto**. Para este trabalho, deixamos as falsas em cerca de **30%** do acervo, que é
> mais perto do que chega à nossa fila do que um meio a meio.
>
> Dois detalhes técnicos que importam. Primeiro: as notícias verdadeiras **foram
> cortadas para ter o mesmo tamanho** da falsa correspondente. Sem isso, um modelo
> aprenderia que "texto curto é falso" e pararia por aí. Segundo: o que o modelo pode
> aprender aqui é **como** as falsas são escritas e **sobre o que** elas falam. Ele não tem
> acesso aos fatos. Nunca vai ter.
>
> — Lúcia Fernandes

---

*Cenário simulado; textos reais (Monteiro et al., 2018, Fake.br Corpus, versão de tamanho
normalizado). As falsas foram subamostradas para 30%.*

## 🎯 A sua missão

Construir um modelo que lê uma notícia e decide: **VERDADEIRA ou FALSA?** A classe
crítica é **falsa** (rótulo 0, cerca de 30% do acervo): uma falsa que passa batida
continua circulando.

Os textos aqui são longos, de jornalismo e de pseudojornalismo, e muito mais parecidos
entre si do que uma avaliação de produto. E o desequilíbrio tem uma armadilha escondida
para um dos três classificadores.

### 🗺️ O mapa da operação

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | Ler notícias e descobrir o que denuncia uma falsa | Só clicando |
| **2 · CONFIGURAR** | Decidir como o texto vira número | Só arrastando |
| **3 · MODELAR** | Vetorizar, treinar e prever | ✏️ Completando **11 lacunas** |
| **4 · AVALIAR** | Medir, ler os termos e testar o estilo | Só clicando |

> ⚠️ **Antes de começar:** vá em `Ambiente de execução` → `Alterar o tipo de ambiente de
> execução` e confirme que está em **CPU**. Tudo aqui roda em segundos, sem placa de vídeo.
>
> Use `Arquivo` → `Salvar uma cópia no Drive` antes de começar, para não perder o trabalho.
>
> 📎 **Simulado de preparação.** Este caderno imita o formato dos desafios oficiais da Fase 1,
> com dados públicos diferentes. O gabarito do teste fica com o corretor
> (`simulados/corrigir.py`), nunca dentro do caderno.

In [ ]:
#@title 🛠️ Preparação: abrindo o acervo { display-mode: "form" }
# Os arquivos treino.csv e teste.csv ficam na mesma pasta deste caderno.
URL_TREINO = ""
URL_TESTE = ""

import pathlib, warnings, re
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

NOME_CLASSE = {0: 'FALSA', 1: 'VERDADEIRA'}
# A classe que custa caro errar. Toda a avaliação do Bloco 4 gira em torno dela.
CLASSE_CRITICA = 0

def _carregar(url, nome_local):
    """Usa a cópia ao lado do caderno; senão tenta o endereço; senão pede o envio."""
    if pathlib.Path(nome_local).exists():
        return pd.read_csv(nome_local)
    if url:
        print(f"Baixando {nome_local}...")
        try:
            return pd.read_csv(url)
        except Exception as erro:
            print(f"  não deu certo ({erro.__class__.__name__}).")
    from google.colab import files
    print(f"  envie o arquivo {nome_local}:")
    files.upload()
    return pd.read_csv(nome_local)

dados = _carregar(URL_TREINO, "treino.csv")
_teste = _carregar(URL_TESTE, "teste.csv")
mensagens_teste = _teste["mensagem"]
IDS_TESTE = _teste["id_mensagem"].to_numpy()
N_TESTE = len(mensagens_teste)

_p0 = (dados.rotulo == 0).mean() * 100
print("\nAmbiente carregado.\n")
print(f"  Acervo público (rotulado) ......... {len(dados):>6} textos")
print(f"     {NOME_CLASSE[0].lower():<14} (rótulo 0) ... {(dados.rotulo==0).sum():>6}   {_p0:4.1f}%")
print(f"     {NOME_CLASSE[1].lower():<14} (rótulo 1) ... {(dados.rotulo==1).sum():>6}   {100-_p0:4.1f}%")
print(f"  Conjunto de teste (sem rótulo) .... {N_TESTE:>6} textos  [tamanho fixo]")
print(f"\n  A classe crítica (0) é a minoritária. Guardem esse número para o Bloco 4.")
for _, r in dados.head(2).iterrows():
    print(f"\n  [{NOME_CLASSE[r.rotulo]}]  {r.mensagem[:140]}...")

---
# 🔍 BLOCO 1 · EXPLORAR
## *Leiam como checadores*

> **Lúcia:** *"Notícia falsa tem cheiro. Adjetivo demais, fonte de menos, pressa para você
> compartilhar. Mas as boas não têm cheiro nenhum."*

Enquanto leem, procurem:

- **Fonte.** A notícia diz quem afirmou o quê? Com nome, cargo, data?
- **Tom.** Há adjetivos de indignação, caixa alta, pedidos de compartilhamento?
- **Assunto.** As falsas se concentram em certos temas e personagens? Guardem isso: um
  modelo que aprende *assunto* em vez de *forma* envelhece rápido.

In [ ]:
#@title 📬 Leia textos do acervo { display-mode: "form" }
#@markdown Escolha a classe e rode a célula (▶). Leia com atenção de investigador.
tipo = "falsa"  #@param ["falsa", "verdadeira"]
quantidade = 6  #@param {type:"slider", min:3, max:12, step:1}
semente_amostra = 7  #@param {type:"integer"}

alvo = 0 if tipo == NOME_CLASSE[0].lower() else 1
amostra = dados[dados.rotulo == alvo].sample(quantidade, random_state=semente_amostra)
print("="*70)
print(f"   {NOME_CLASSE[alvo]}  ·  {quantidade} textos do acervo")
print("="*70)
for k, (_, r) in enumerate(amostra.iterrows(), 1):
    texto = r.mensagem if len(r.mensagem) < 600 else r.mensagem[:600] + " [...]"
    print(f"\n[{k}] {texto}")
print("\n" + "="*70)
print("Troquem a semente para ler outros. Anotem os marcadores que aparecem SÓ de um lado.")

In [ ]:
#@title 📊 Investigue os dados: escolha a variável E o tipo de gráfico { display-mode: "form" }
variavel = "n_palavras"  #@param ["n_palavras", "prop_maiusculas", "n_exclamacoes", "cita_fonte", "tem_numero", "tem_aspas", "apelo_emocional"]
tipo_de_grafico = "Boxplot"  #@param ["Histograma sobreposto", "Boxplot", "Barras de proporcao", "Termos mais frequentes por classe"]

dados["n_palavras"] = dados.mensagem.str.split().str.len()
dados["prop_maiusculas"] = dados.mensagem.str.count(r"[A-ZÁÉÍÓÚÂÊÔÃÕÇ]") / dados.mensagem.str.count(r"[A-Za-zÀ-ÿ]").clip(lower=1)
dados["n_exclamacoes"] = dados.mensagem.str.count("!")
dados["cita_fonte"] = dados.mensagem.str.contains(r"segundo|de acordo com|afirmou|informou|disse", case=False).astype(int)
dados["tem_numero"] = dados.mensagem.str.contains(r"\d").astype(int)
dados["tem_aspas"] = dados.mensagem.str.contains("[\"“”]").astype(int)
dados["apelo_emocional"] = dados.mensagem.str.contains(r"urgente|absurd|compartilh|vergonha|chocante|bomba", case=False).astype(int)

CORES_C = ["#E6177A", "#3DCB55"]
serie = dados[variavel]
eh_binaria = serie.nunique() <= 3
rot_leg = [NOME_CLASSE[0].capitalize(), NOME_CLASSE[1].capitalize()]
limite = None if eh_binaria else float(serie.quantile(0.99))
vista = serie if eh_binaria else serie.clip(upper=limite)
grupos = [vista[dados.rotulo == k] for k in [0, 1]]

if (tipo_de_grafico == "Barras de proporcao") != eh_binaria and tipo_de_grafico != "Termos mais frequentes por classe":
    print(f"[!] {tipo_de_grafico} não é a melhor escolha para uma variável "
          f"{'binária' if eh_binaria else 'contínua'}. O gráfico sai assim mesmo.\n")

if tipo_de_grafico == "Termos mais frequentes por classe":
    from sklearn.feature_extraction.text import CountVectorizer
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax_, k, cor in zip(axes, [0, 1], CORES_C):
        cv = CountVectorizer(min_df=5, max_features=3000, token_pattern=r"(?u)\b\w\w\w+\b")
        m = cv.fit_transform(dados.mensagem[dados.rotulo == k])
        freq = pd.Series(np.asarray(m.sum(axis=0)).ravel(), index=cv.get_feature_names_out())
        top = freq.sort_values().tail(15)
        ax_.barh(top.index, top.values, color=cor)
        ax_.set_title(f"Mais frequentes: {NOME_CLASSE[k]}", fontweight="bold")
    plt.tight_layout(); plt.show()
    print("Os termos mais FREQUENTES quase sempre aparecem dos dois lados. Frequência não")
    print("é poder discriminante: é por isso que existe o IDF.")
else:
    fig, ax = plt.subplots(figsize=(8.5, 4.3))
    if tipo_de_grafico == "Histograma sobreposto":
        bins = np.histogram_bin_edges(vista, bins=30)
        for g, cor, r in zip(grupos, CORES_C, rot_leg):
            ax.hist(g, bins=bins, alpha=.55, color=cor, label=r, density=True, edgecolor="white")
        ax.legend(); ax.set_ylabel("densidade")
    elif tipo_de_grafico == "Boxplot":
        bp = ax.boxplot(grupos, patch_artist=True, tick_labels=rot_leg,
                        medianprops=dict(color="black", linewidth=2))
        for p, cor in zip(bp["boxes"], CORES_C): p.set_facecolor(cor); p.set_alpha(.8)
    else:
        prop = [g.mean() for g in grupos]
        ax.bar(rot_leg, prop, color=CORES_C, edgecolor="#1F2A44")
        ax.set_ylabel("proporção com a característica")
        for i, v in enumerate(prop): ax.text(i, v, f"{v*100:.1f}%", ha="center", va="bottom")
    ax.set_title(f"'{variavel}' por classe", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    print(f"Médias: {rot_leg[0]} = {serie[dados.rotulo==0].mean():.3f}  |  "
          f"{rot_leg[1]} = {serie[dados.rotulo==1].mean():.3f}")

> **Lúcia:** *"Repararam que as verdadeiras terminam no meio da frase? É o corte de
> tamanho. Sem ele, vocês estariam medindo a régua, não a notícia."*

---
# ⚙️ BLOCO 2 · CONFIGURAR
## *Como o texto vira número*

| Conjunto | Tamanho | Vocês têm os rótulos? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 4.442 textos | Sim | Treinar o modelo e validar as escolhas |
| **Teste** | **700 (fixo)** | Não | Comparação final (corrigida por `simulados/corrigir.py`) |

> Os valores iniciais dos controles abaixo são **deliberadamente modestos**: produzem um
> modelo que funciona e que está longe do que o dado permite.

In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
tamanho_da_validacao = 25  #@param {type:"slider", min:10, max:40, step:5}
max_palavras = 400  #@param {type:"slider", min:200, max:50000, step:200}
ngramas = "so 1 palavra"  #@param ["so 1 palavra", "1 a 2 palavras", "1 a 3 palavras"]
min_df = 3  #@param {type:"slider", min:1, max:20, step:1}
modelo_escolhido = "Naive Bayes"  #@param ["Regressao Logistica", "SVM Linear", "Naive Bayes"]
forca_C = 1.0  #@param {type:"slider", min:0.01, max:20, step:0.01}

FAIXA = {"so 1 palavra": (1, 1), "1 a 2 palavras": (1, 2), "1 a 3 palavras": (1, 3)}
faixa_ngrama = FAIXA[ngramas]

print("PARÂMETROS DEFINIDOS")
print(f"   Validação .......... {tamanho_da_validacao}% do conjunto público")
print(f"   Vocabulário ........ até {max_palavras} termos")
print(f"   N-gramas ........... {ngramas}  {faixa_ngrama}")
print(f"   min_df ............. {min_df}")
print(f"   Classificador ...... {modelo_escolhido}")
print(f"   Força C ............ {forca_C}")
print(f"   Teste .............. {N_TESTE} textos  [FIXO]")

### As decisões e seus efeitos

**Vocabulário e n-gramas.** Notícias são longas e ricas em vocabulário. Expressões como
"de acordo com" ou "nesta terça-feira" só existem como unidade a partir de `(1, 2)` ou
`(1, 3)`.

**Desequilíbrio e o Naive Bayes.** O Naive Bayes multiplica a probabilidade de cada termo
pela **proporção das classes no treino**. Com 70% de verdadeiras e textos longos, cheios
de termos, essa multiplicação pode empurrar quase tudo para a classe majoritária. Testem
o Naive Bayes com vocabulário grande e olhem o recall de falsa. Depois procurem na tabela
o parâmetro que muda a proporção assumida.

### 📗 Referência técnica: nome exato e parâmetros de cada classe

**Vocês vão precisar desta tabela no Bloco 3.**

> No Bloco 3, apenas alguns parâmetros aparecem preenchidos. Os demais assumem
> silenciosamente o valor padrão. Estudar esta tabela e **acrescentar parâmetros à chamada**
> produz resultados melhores. A diferença é intencional.

#### 🔤 Vetorizador TF-IDF

```python
from sklearn.feature_extraction.text import TfidfVectorizer
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `max_features` | `None` | Limita o vocabulário aos termos mais frequentes |
| `ngram_range` | `(1, 1)` | Faixa de n-gramas. `(1, 2)` inclui bigramas |
| `min_df` | `1` | Frequência documental mínima |
| `max_df` | `1.0` | Descarta termos presentes em mais que essa fração dos documentos |
| `lowercase` | `True` | Converte tudo para minúsculas |
| `strip_accents` | `None` | `'unicode'` remove acentos e unifica grafias |
| `sublinear_tf` | `False` | **`True` aplica escala logarítmica à frequência e costuma ajudar** |
| `stop_words` | `None` | Lista de palavras a descartar |
| `token_pattern` | `r"(?u)\b\w\w+\b"` | O padrão **descarta palavras de uma letra** |

#### 📈 Regressão Logística

```python
from sklearn.linear_model import LogisticRegression
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `C` | `1.0` | Inverso da regularização. **Maior = menos regularização** |
| `max_iter` | `100` | Iterações máximas. Com vocabulário grande, frequentemente insuficiente |
| `class_weight` | `None` | `'balanced'` ou um dicionário, como `{0: 1.5, 1: 1}` |
| `solver` | `'lbfgs'` | Algoritmo de otimização |

#### ⚔️ SVM Linear

```python
from sklearn.svm import LinearSVC
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `C` | `1.0` | Penalidade por erro. Em texto, valores **abaixo de 1** costumam ir melhor |
| `class_weight` | `None` | `'balanced'` ou um dicionário, como `{0: 1.5, 1: 1}` |
| `max_iter` | `1000` | Iterações máximas |

#### 🎲 Naive Bayes Multinomial

```python
from sklearn.naive_bayes import MultinomialNB
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `alpha` | `1.0` | Suavização: evita probabilidade zero para termo nunca visto numa classe |
| `fit_prior` | `True` | Aprende a proporção das classes a partir do treino |
| `class_prior` | `None` | **Fixa** a proporção assumida, por exemplo `[0.5, 0.5]` |

> ⚠️ O Naive Bayes **não aceita `class_weight`**. O ajuste equivalente para o desequilíbrio
> é o `class_prior`.

---
# ✏️ BLOCO 3 · MODELAR
## *Dois estimadores encadeados*

**1. O TRADUTOR (o vetorizador TF-IDF)** — transforma texto em números.

| Comando | O que faz |
|:---|:---|
| `.fit_transform(textos)` | **APRENDE** o vocabulário e o IDF E traduz o texto |
| `.transform(textos)` | **SÓ TRADUZ**, com o vocabulário que já aprendeu |

**2. O CLASSIFICADOR** — decide a classe.

| Comando | O que faz |
|:---|:---|
| `.fit(X, y)` | **TREINA** |
| `.predict(X)` | **PREVÊ** |

### O ponto crítico: vazamento de dados

> **Por que o vetorizador usa `fit_transform` no treino e só `transform` na validação?**

O `fit` do vetorizador estima parâmetros: o vocabulário e o IDF de cada termo, que depende
de em quantos documentos ele aparece. Se o `fit` incluir a validação, esses parâmetros
carregam informação de documentos que deveriam ser desconhecidos. Nenhum erro aparece, o
código roda, e a métrica **melhora** — falsamente. A verificação deste bloco detecta isso.

`else` não recebe condição, apenas os dois-pontos.

> ⚠️ A célula não executa enquanto houver lacunas.

In [ ]:
# ✏️ SUA VEZ · Parte 1: separar os dados e construir as ferramentas
# Substituam cada ____ pelo que falta. A célula só executa quando não sobrar nenhum.
X = dados["mensagem"]
y = dados["rotulo"]

# LACUNA 1 ▸ 'stratify' precisa receber o vetor cuja proporção deve ser preservada.
#            É o texto ou o vetor de rótulos?
X_treino, X_valida, y_treino, y_valida = train_test_split(
    X, y, test_size=tamanho_da_validacao/100, random_state=42, stratify=____)

# LACUNA 2 ▸ o nome exato da classe que transforma texto em TF-IDF está no Bloco 2.
vetorizador = ____(max_features=max_palavras, ngram_range=faixa_ngrama, min_df=min_df)

# LACUNAS 3 a 7 ▸ os nomes exatos dos classificadores e as duas palavras da cadeia.
if modelo_escolhido == "Regressao Logistica":
    modelo = ____(C=forca_C, max_iter=1000, random_state=42)

____ modelo_escolhido == "SVM Linear":
    modelo = ____(C=forca_C, random_state=42)

____:
    modelo = ____()

# ZONA LIVRE ▸ acrescentem parâmetros às chamadas acima (tabelas do Bloco 2).

#   Nesta operação, testem o peso das classes: 'class_weight' na Regressão Logística e
#   no SVM, e 'class_prior' no Naive Bayes (que não aceita class_weight). No vetorizador,
#   'sublinear_tf' costuma ajudar muito com textos longos. Um de cada vez.

print(f"Treino:    {len(X_treino)} textos")
print(f"Validação: {len(X_valida)} textos")
print(f"Vetorizador: {type(vetorizador).__name__}  |  Classificador: {type(modelo).__name__}")

In [ ]:
# ✏️ SUA VEZ · Parte 2: vetorizar, treinar e prever
# Atenção: as lacunas 8 e 9 são as duas de nome parecido. Releiam o Bloco 3 antes.

# LACUNA 8 ▸ no TREINO, o vetorizador precisa APRENDER o vocabulário e o IDF, e
#            também devolver a matriz transformada. Qual método faz as duas coisas?
X_treino_vec = vetorizador.____(X_treino)

# LACUNA 9 ▸ na VALIDAÇÃO, ele deve apenas APLICAR o que já aprendeu. Qual método?
X_valida_vec = vetorizador.____(X_valida)

# LACUNA 10 ▸ qual método ajusta o classificador ao conjunto de treino?
modelo.____(X_treino_vec, y_treino)

# LACUNA 11 ▸ qual método infere as classes de textos que o modelo nunca viu?
previsoes = modelo.____(X_valida_vec)

print(f"Vocabulário aprendido: {len(vetorizador.vocabulary_)} termos")
print(f"Previsões geradas: {len(previsoes)}")

In [ ]:
#@title 🔎 Verificação das lacunas (inclusive vazamento) { display-mode: "form" }
erros = []
try:
    esperado = np.bincount(y, minlength=2) * len(y_treino) / len(y)
    desvio = np.abs(np.bincount(y_treino, minlength=2) - esperado).max()
    if desvio > 5:
        erros.append(f"LACUNA 1: o treino ficou {desvio:.0f} exemplos fora da composição "
                     "de um split estratificado. O 'stratify' não recebeu o vetor correto.")
except NameError:
    erros.append("LACUNA 1: X_treino/y_treino não existem.")
try:
    if type(vetorizador).__name__ != "TfidfVectorizer":
        erros.append(f"LACUNA 2: esperava TfidfVectorizer, obteve {type(vetorizador).__name__}.")
except NameError:
    erros.append("LACUNA 2: a variável 'vetorizador' não existe.")
try:
    esp = {"Regressao Logistica": "LogisticRegression", "SVM Linear": "LinearSVC",
           "Naive Bayes": "MultinomialNB"}[modelo_escolhido]
    if type(modelo).__name__ != esp:
        erros.append(f"LACUNAS 3 a 7: para '{modelo_escolhido}' era esperada a classe {esp}, "
                     f"mas foi construída {type(modelo).__name__}.")
except NameError:
    erros.append("LACUNAS 3 a 7: a variável 'modelo' não existe.")

vocab_atual = set()
try: vocab_atual = set(vetorizador.vocabulary_.keys())
except Exception:
    erros.append("LACUNA 8: o vetorizador não tem parâmetros estimados.")
if vocab_atual:
    # Referência: um vetorizador com os MESMOS parâmetros, ajustado só no treino.
    _v = TfidfVectorizer(**vetorizador.get_params()).fit(X_treino)
    if vocab_atual != set(_v.vocabulary_.keys()):
        erros.append("LACUNA 9: VAZAMENTO DE DADOS DETECTADO. O vocabulário difere do que "
                     "se obtém ajustando só no treino: a validação participou da estimação. "
                     "Nenhuma exceção foi lançada e a métrica ficaria melhor — é assim que "
                     "esse erro passa despercebido.")
try:
    if X_valida_vec.shape[1] != X_treino_vec.shape[1]:
        erros.append("LACUNAS 8 e 9: treino e validação com números de colunas diferentes.")
except NameError:
    erros.append("LACUNAS 8 ou 9: faltou criar X_treino_vec ou X_valida_vec.")
try:
    if not hasattr(modelo, "classes_"):
        erros.append("LACUNA 10: o classificador não foi ajustado.")
except NameError: pass
try:
    if len(previsoes) != len(X_valida):
        erros.append(f"LACUNA 11: {len(previsoes)} previsões para {len(X_valida)} textos.")
except NameError:
    erros.append("LACUNA 11: a variável 'previsoes' não existe.")

if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
else:
    print("VERIFICAÇÃO APROVADA\n")
    print("   Split estratificado, vetorizador ajustado só no treino, modelo treinado.")
    _ext = {k: v for k, v in modelo.get_params().items()
            if k in ("class_weight", "class_prior", "alpha", "fit_prior")
            and v not in (None, 1.0, True)}
    _vext = {k: v for k, v in vetorizador.get_params().items()
             if k in ("sublinear_tf", "strip_accents", "max_df", "stop_words")
             and v not in (None, False, 1.0)}
    if _ext or _vext:
        print(f"\n   Parâmetros adicionais na ZONA LIVRE: {_ext} {_vext}")

---
# 📈 BLOCO 4 · AVALIAR
## *Quantas falsas continuam circulando?*

> **Lúcia:** *"Se o sistema marcar uma verdadeira como falsa, um checador perde meia hora.
> Se deixar passar uma falsa, ela chega a um milhão de pessoas antes de a gente ver."*

Métrica oficial: **F1-macro**. Métrica de apoio: **recall da falsa** — das notícias
falsas, quantas o modelo mandou para a fila de checagem.

In [ ]:
#@title 🏆 O resultado { display-mode: "form" }
acuracia = accuracy_score(y_valida, previsoes)
f1 = f1_score(y_valida, previsoes, average="macro")
recall_critica = recall_score(y_valida, previsoes, pos_label=CLASSE_CRITICA)
precisao_critica = precision_score(y_valida, previsoes, pos_label=CLASSE_CRITICA,
                                   zero_division=0)
_nc = NOME_CLASSE[CLASSE_CRITICA].upper()

print("="*64)
print("             DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*64)
print(f"  Acurácia ...................................... {acuracia*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) .................. {f1*100:5.1f}%")
print(f"  🔴 RECALL DE {_nc:<14} (métrica de apoio) ..... {recall_critica*100:5.1f}%")
print(f"     Precisão de {_nc.lower():<14} ................... {precisao_critica*100:5.1f}%")
print("="*64)
print(f"  Referência: responder sempre a classe 1 daria {(y_valida==1).mean()*100:.1f}% de "
      "acurácia e F1-macro perto de 41%.")
if   f1 >= .89: print("\n  Desempenho alto. Avaliem o recall da classe crítica e o risco de sobreajuste.")
elif f1 >= .86: print("\n  Desempenho sólido. Examinem o recall da classe crítica.")
elif f1 >= .80: print("\n  Desempenho intermediário. Revisem n-gramas, vocabulário e modelo.")
else:           print("\n  Desempenho baixo. A representação atual não separa as classes.")
if recall_critica < .3:
    print(f"\n[!] O modelo quase não aponta a classe {_nc}. Ele 'desistiu' da classe")
    print("    minoritária. Revejam o Bloco 2: existe um parâmetro para isso em cada modelo.")

cm = confusion_matrix(y_valida, previsoes, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=[NOME_CLASSE[0], NOME_CLASSE[1]]).plot(
    ax=ax, cmap="viridis", colorbar=False)
ax.set_title("Onde o modelo acertou e onde falhou", fontweight="bold")
ax.set_xlabel("O que o modelo DISSE"); ax.set_ylabel("O que REALMENTE era")
plt.tight_layout(); plt.show()
print(f"\n   {_nc} NÃO DETECTADOS ........ {cm[0,1]}   (erro de custo alto)")
print(f"   Falsos alarmes ............... {cm[1,0]}   (erro de custo baixo, mas não nulo)\n")
print(classification_report(y_valida, previsoes, labels=[0, 1],
      target_names=[NOME_CLASSE[0], NOME_CLASSE[1]], digits=3, zero_division=0))

In [ ]:
#@title 🔤 Os termos que mais pesam { display-mode: "form" }
# Em problema binário, coef_[0] é o peso a favor da classe 1:
# coeficiente negativo puxa para a classe 0 (a crítica).
if hasattr(modelo, "coef_"):
    termos = np.array(vetorizador.get_feature_names_out())
    pesos = modelo.coef_[0]
    ordem = np.argsort(pesos)
    top0, val0 = termos[ordem[:12]], np.abs(pesos[ordem[:12]])
    top1, val1 = termos[ordem[-12:]][::-1], pesos[ordem[-12:]][::-1]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].barh(range(12), val0[::-1], color="#E6177A")
    axes[0].set_yticks(range(12)); axes[0].set_yticklabels(top0[::-1])
    axes[0].set_title(f"🔴 Termos que mais indicam {NOME_CLASSE[0].upper()}", fontweight="bold")
    axes[1].barh(range(12), val1[::-1], color="#3DCB55")
    axes[1].set_yticks(range(12)); axes[1].set_yticklabels(top1[::-1])
    axes[1].set_title(f"🟢 Termos que mais indicam {NOME_CLASSE[1].upper()}", fontweight="bold")
    for a_ in axes: a_.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    print("Comparem com as hipóteses do Bloco 1. O modelo confirmou, refutou, ou achou")
    print("marcadores que vocês não tinham previsto?")
else:
    print("ℹ️ O Naive Bayes não expõe pesos da mesma forma.")
    print("   Escolham Regressão Logística ou SVM Linear no Bloco 2 para ver os termos.")

In [ ]:
#@title 🎭 Mesmo fato, outro estilo { display-mode: "form" }
#@markdown Dois textos com o MESMO conteúdo e estilos diferentes. Editem e rodem de novo.
texto_sobrio = "O ministro da Economia afirmou nesta terça-feira que o governo vai enviar ao Congresso uma proposta de reforma tributária até o fim do mês, segundo nota divulgada pela assessoria do ministério."  #@param {type:"string"}
texto_alarmista = "URGENTE!!! O ministro da Economia CONFESSOU que o governo vai enviar ao Congresso uma reforma tributária ABSURDA até o fim do mês!!! Compartilhe antes que apaguem!!!"  #@param {type:"string"}

for rotulo_txt, t in [("SÓBRIO", texto_sobrio), ("ALARMISTA", texto_alarmista)]:
    v = vetorizador.transform([t])
    palpite = modelo.predict(v)[0]
    linha = f"   {rotulo_txt:<10} → {NOME_CLASSE[palpite]}"
    if hasattr(modelo, "decision_function"):
        linha += f"   (escore {modelo.decision_function(v)[0]:+.2f}; negativo puxa para FALSA)"
    print(linha)
print("\nO fato é o mesmo nos dois textos. Se a resposta mudou, o modelo reagiu ao ESTILO.")
print("Agora o teste inverso: escrevam uma afirmação falsa com o tom mais sóbrio que")
print("conseguirem, citando uma fonte. O modelo percebe? O que isso diz sobre usar este")
print("modelo para decidir, sozinho, o que é mentira?")

---
# 🤔 PARADA OBRIGATÓRIA: o modelo não sabe o que é verdade

> **Lúcia:** *"Já me ofereceram um sistema que 'detecta fake news' para derrubar posts
> automaticamente. Eu recusei. Quero que vocês me digam se eu fiz bem."*

### Questões para a equipe deliberar

1. **Estilo não é verdade.** O modelo nunca viu um fato: viu palavras. A célula do estilo
   mostrou o que acontece quando se muda só o tom. Que tipo de notícia falsa este modelo
   **não** consegue pegar? E que tipo de notícia verdadeira ele corre o risco de marcar
   como falsa (sátira, opinião, veículos pequenos com linguagem popular)?

2. **Triagem ou censura.** Usar o modelo para ordenar a fila dos checadores e usá-lo para
   remover publicações automaticamente são coisas muito diferentes, com o mesmo modelo.
   Qual restrição de uso vocês consideram inegociável?

3. **Envelhecimento.** O acervo é de 2016 a 2018: outros personagens, outros assuntos.
   Numa eleição em 2026, o que acontece com os termos de maior peso? Como a agência
   perceberia a degradação sem ter o rótulo?

4. **Pareamento.** O acervo pareou cada falsa com uma verdadeira do mesmo assunto. Por
   que isso importa para o que o modelo aprende? O que aconteceria sem o pareamento?

In [ ]:
#@title 📝 O relatório da sua equipe { display-mode: "form" }
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. O modelo aprendeu a reconhecer mentira ou estilo? O que a célula do estilo mostrou?**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. Quais termos mais indicam notícia falsa? Algum surpreendeu?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. Que restrição de uso vocês consideram inegociável, e por quê?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. Como a agência perceberia que o modelo envelheceu, sem ter o rótulo?**
resposta_4 = ""  #@param {type:"string"}

relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido,
             "max_palavras": max_palavras, "ngramas": ngramas, "min_df": min_df,
             "C": forca_C, "validacao_pct": tamanho_da_validacao,
             "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "recall_critica": round(recall_critica, 4),
             "precisao_critica": round(precisao_critica, 4),
             "just_estilo": resposta_1, "just_termos": resposta_2, "just_restricao_de_uso": resposta_3, "just_deriva": resposta_4}
faltando = [k for k in ["equipe", "just_estilo", "just_termos", "just_restricao_de_uso", "just_deriva"]
            if not str(relatorio[k]).strip()]
if faltando:
    print("⚠️ Ainda faltam campos:", ", ".join(faltando))
else:
    print("Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
import json, pathlib

PASTA_SUBMISSAO = pathlib.Path("submissoes")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido, "f1_macro": round(f1, 4)}

# Repare: só transform. Coerência com a regra do Bloco 3.
palpites = modelo.predict(vetorizador.transform(mensagens_teste))
submissao = pd.DataFrame({"id_mensagem": IDS_TESTE, "rotulo_previsto": palpites})

assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_checagem.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_checagem.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_checagem.csv   ({len(submissao)} linhas)")
print("  relatorio_checagem.json")
print("="*60)
_n0 = int((palpites == CLASSE_CRITICA).sum())
print(f"\n  Marcados como {NOME_CLASSE[CLASSE_CRITICA].lower()}: {_n0} de {len(palpites)} "
      f"({_n0/len(palpites)*100:.1f}%). No acervo: {(dados.rotulo==0).mean()*100:.1f}%.")
display(submissao.head())
print("\n  Para corrigir (da raiz do repositório):")
print(f"     python simulados/corrigir.py checagem {csv_saida}")

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA OPERAÇÃO CHECAGEM

> **Lúcia:** *"O modelo de vocês agora ordena a fila da manhã. Nada é derrubado por ele,
> nada é publicado por ele: ele só decide o que a equipe lê primeiro. Na semana passada
> pegamos uma corrente falsa duas horas mais cedo. E quando ele erra, o erro fica numa
> fila, não na vida de ninguém."*

---

### Competências praticadas neste simulado

- TF-IDF em textos longos e `sublinear_tf`
- O colapso do Naive Bayes sob desequilíbrio e o papel do `class_prior`
- **Atalhos** (*shortcut learning*): por que o corpus normalizou o tamanho dos textos
- Estilo × conteúdo: o que um classificador de texto realmente aprende
- Deriva temporal de temas e personagens
- Triagem assistida × remoção automática: o mesmo modelo, dois usos